In [2]:
import random
from collections import defaultdict

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
# -------------------------------
# Step 1: Dataset
# -------------------------------
text = """
artificial intelligence is transforming modern society.
it is used in healthcare finance education and transportation.
machine learning allows systems to improve automatically with experience.
data plays a critical role in training intelligent systems.
large datasets help models learn complex patterns.
deep learning uses multi layer neural networks.
neural networks are inspired by biological neurons.
each neuron processes input and produces an output.
training a neural network requires optimization techniques.
gradient descent minimizes the loss function.

natural language processing helps computers understand human language.
text generation is a key task in nlp.
language models predict the next word or character.
recurrent neural networks handle sequential data.
lstm and gru models address long term dependency problems.
however rnn based models are slow for long sequences.

transformer models changed the field of nlp.
they rely on self attention mechanisms.
attention allows the model to focus on relevant context.
transformers process data in parallel.
this makes training faster and more efficient.
modern language models are based on transformers.

education is being improved using artificial intelligence.
intelligent tutoring systems personalize learning.
automated grading saves time for teachers.
online education platforms use recommendation systems.
technology enhances the quality of learning experiences.

ethical considerations are important in artificial intelligence.
fairness transparency and accountability must be ensured.
ai systems should be designed responsibly.
data privacy and security are major concerns.
researchers continue to improve ai safety.

text generation models can create stories poems and articles.
they are used in chatbots virtual assistants and content creation.
generated text should be meaningful and coherent.
evaluation of text generation is challenging.
human judgement is often required.

continuous learning is essential in the field of ai.
research and innovation drive technological progress.
students should build strong foundations in mathematics.
programming skills are important for ai engineers.
practical experimentation enhances understanding.

"""


In [4]:

# -------------------------------
# Step 2: Preprocessing
# -------------------------------
def preprocess_text(text):
    text = text.lower()
    text = text.replace('\n', ' ')
    tokens = text.split()
    return tokens

tokens = preprocess_text(text)

In [5]:
# -------------------------------
# Step 3: Build N-grams
# -------------------------------
def build_ngrams(tokens, n):
    ngrams = defaultdict(list)
    for i in range(len(tokens) - n):
        context = tuple(tokens[i:i+n])
        next_word = tokens[i+n]
        ngrams[context].append(next_word)
    return ngrams

N = 2   # Bigram model
ngram_model = build_ngrams(tokens, N)


In [6]:
# -------------------------------
# Step 4: Text Generation
# -------------------------------
def generate_text(model, seed, n, length=20):
    current = tuple(seed.split())
    result = list(current)

    for _ in range(length):
        if current not in model:
            break
        next_word = random.choice(model[current])
        result.append(next_word)
        current = tuple(result[-n:])

    return " ".join(result)


In [9]:
# -------------------------------
# Step 5: Generate Output N-gram
# -------------------------------
seed_text = "artificial intelligence"
generated_text = generate_text(ngram_model, seed_text, N, length=15)

print("Generated Text:")
print(generated_text)

Generated Text:
artificial intelligence is transforming modern society. it is used in chatbots virtual assistants and content creation. generated


In [10]:
# -------------------------------
# Step 2: Tokenization
# -------------------------------
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

token_list = tokenizer.texts_to_sequences([text])[0]


In [11]:
# -------------------------------
# Step 3: Create Input Sequences
# -------------------------------
input_sequences = []

for i in range(1, len(token_list)):
    n_gram_sequence = token_list[:i+1]
    input_sequences.append(n_gram_sequence)

max_len = max(len(seq) for seq in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding='pre')

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)


In [12]:
# -------------------------------
# Step 4: Build LSTM Model
# -------------------------------
model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [16]:
def sample_with_temperature(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")
    preds = np.log(preds + 1e-8) / temperature
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    return np.random.choice(len(preds), p=preds)

def generate_text(seed_text, next_words, model, max_len, temperature=0.8):
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([seed_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_len-1, padding='pre')

        preds = model.predict(sequence, verbose=0)[0]
        predicted_index = sample_with_temperature(preds, temperature)

        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                seed_text += " " + word
                break

    return seed_text

In [17]:
model.fit(X, y, epochs=150, verbose=1)


Epoch 1/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 0.1914
Epoch 2/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 1.0000 - loss: 0.1848
Epoch 3/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.1843
Epoch 4/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.1860
Epoch 5/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.1798
Epoch 6/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.1767
Epoch 7/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.1688
Epoch 8/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.1707
Epoch 9/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.1633
Epoch 10/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.1540
Epoch 11/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 1.0000 - loss: 0.1543
Epoch 12/150
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step

In [18]:
# LSTM
print(generate_text(seed_text, 20, model, max_len, temperature=0.6))
print(generate_text(seed_text, 20, model, max_len, temperature=0.8))
print(generate_text(seed_text, 20, model, max_len, temperature=1.0))


artificial intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically
artificial intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically
artificial intelligence is transforming modern society society it is used in healthcare finance education and transportation machine learning allows systems to improve


In [19]:
print(generate_text(seed_text, 20, model, max_len, temperature=0.8))

artificial intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning allows systems to improve automatically


In [20]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [22]:
# -------------------------------
# Step 4: Build RNN Model
# -------------------------------
model = Sequential()
model.add(Embedding(total_words, 50, input_length=max_len-1))
model.add(SimpleRNN(64))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


In [23]:
# -------------------------------
# Step 5: Train Model
# -------------------------------
model.fit(X, y, epochs=200, verbose=1)

# -------------------------------
# Step 6: Text Generation
# -------------------------------
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        sequence = tokenizer.texts_to_sequences([seed_text])[0]
        sequence = pad_sequences([sequence], maxlen=max_len-1, padding='pre')
        predicted = np.argmax(model.predict(sequence, verbose=0), axis=-1)

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break
    return seed_text

# -------------------------------
# Step 7: Output
# -------------------------------
seed = "artificial intelligence"
print("Generated Text:")
print(generate_text(seed, 15))

Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 5s 144ms/step - accuracy: 0.0022 - loss: 5.2852
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0278 - loss: 5.2024
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.0922 - loss: 5.1162
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.1524 - loss: 5.0676
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.1869 - loss: 4.9965
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2720 - loss: 4.9101
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.2977 - loss: 4.8309
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.3820 - loss: 4.7282
Epoch 9/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.4120 - loss: 4.6322
Epoch 10/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.4217 - loss: 4.5208
Epoch 11/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.5660 - loss: 4.4122
Epoch 12/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/ste

In [24]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Embedding, Dense, LayerNormalization,
    Dropout, MultiHeadAttention
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [25]:
# -------------------------------
# Step 4: Positional Encoding
# -------------------------------
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, np.newaxis]
    i = np.arange(d_model)[np.newaxis, :]
    angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
    angle_rads = pos * angle_rates

    angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])
    angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])

    return tf.cast(angle_rads, dtype=tf.float32)


In [26]:
# -------------------------------
# Step 5: Transformer Encoder Block
# -------------------------------
def transformer_encoder(inputs, head_size, num_heads, ff_dim):
    attention = MultiHeadAttention(
        key_dim=head_size,
        num_heads=num_heads
    )(inputs, inputs)

    attention = Dropout(0.1)(attention)
    x = LayerNormalization(epsilon=1e-6)(inputs + attention)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(0.1)(ffn)

    return LayerNormalization(epsilon=1e-6)(x + ffn)


In [27]:
# -------------------------------
# Step 6: Build Transformer Model
# -------------------------------
embed_dim = 64
inputs = Input(shape=(max_len-1,))
embedding = Embedding(total_words, embed_dim)(inputs)

pos_encoding = positional_encoding(max_len-1, embed_dim)
x = embedding + pos_encoding

x = transformer_encoder(x, head_size=64, num_heads=2, ff_dim=128)

x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = Dense(total_words, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


In [33]:
def top_k_sampling(preds, k=5):
    preds = np.squeeze(preds)
    indices = np.argsort(preds)[-k:]
    probs = preds[indices]
    probs = probs / np.sum(probs)
    return np.random.choice(indices, p=probs)


In [34]:
def generate_text(seed_textt, next_words):
    for _ in range(next_words):
        seq = tokenizer.texts_to_sequences([seed_textt])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding='pre')

        preds = model.predict(seq, verbose=0)
        next_index = top_k_sampling(preds, k=5)

        for word, index in tokenizer.word_index.items():
            if index == next_index:
                seed_textt += " " + word
                break

    return seed_textt


In [35]:
model.fit(X, y, epochs=200, verbose=1)

Epoch 1/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8448 - loss: 0.6033
Epoch 2/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8846 - loss: 0.4673
Epoch 3/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8500 - loss: 0.5980
Epoch 4/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8757 - loss: 0.5280
Epoch 5/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8792 - loss: 0.5379
Epoch 6/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8795 - loss: 0.4967
Epoch 7/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8672 - loss: 0.5572
Epoch 8/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8613 - loss: 0.5335
Epoch 9/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9029 - loss: 0.4189
Epoch 10/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8867 - loss: 0.4531
Epoch 11/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8874 - loss: 0.5054
Epoch 12/200
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step -

In [38]:
# -------------------------------
# Step 9: Output TRANSFORMER
# -------------------------------
seed_textt = "artificial intelligence"
print("Generated Text:")
print(generate_text(seed_textt, 15))

Generated Text:
artificial intelligence is transforming modern society it is used in healthcare finance education and transportation machine learning
